# nnUNet Pipeline — Colab GPU Runner

Dataset: **Dataset777_GCEF** | Trainer: **nnUNetTrainer_betterIgnoreSampling** | Config: **3d_fullres**

Prerequisites (run locally before this notebook):
1. `preprocessing_nnUNet_train.py` → produces `Dataset777_GCEF/` in `nnUNet_raw`
2. (For inference) `preprocessing_nnUNet_predict_tif.py` + `preprocessing_nnUNet_predict_split.py` → produces split chunks

Upload the `Dataset777_GCEF/` folder to Google Drive before starting.

## 1) Runtime Setup

In [ ]:
# Verify GPU runtime
!nvidia-smi

In [ ]:
# Install nnUNet v2 and dependencies
!pip -q install nnunetv2

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repository (or skip if already on Drive)
import os
REPO_DIR = '/content/nnUNet4SoilXrayCT'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/agrorony/nnUNet4SoilXrayCT.git {REPO_DIR}
print('Repo dir:', os.listdir(REPO_DIR))

## 2) Register Custom Trainer

In [ ]:
import shutil
import nnunetv2

# Find nnunetv2 trainers directory
nnunet_trainers_dir = os.path.join(
    os.path.dirname(nnunetv2.__file__),
    'training', 'nnUNetTrainer', 'variants', 'sampling'
)
os.makedirs(nnunet_trainers_dir, exist_ok=True)

src = os.path.join(REPO_DIR, 'nnUNetTrainer_betterIgnoreSampling.py')
dst = os.path.join(nnunet_trainers_dir, 'nnUNetTrainer_betterIgnoreSampling.py')
shutil.copy2(src, dst)

# Verify import
from nnunetv2.training.nnUNetTrainer.variants.sampling.nnUNetTrainer_betterIgnoreSampling import nnUNetTrainer_betterIgnoreSampling
print('Custom trainer registered:', nnUNetTrainer_betterIgnoreSampling.__name__)

## 3) Set Environment Variables & Paths

In [ ]:
import os

# Root folder on Drive (mirrors local G:\האחסון שלי\soil_microCT_images\nnUNet_resources\bnei_reem)
DRIVE_BASE = '/content/drive/MyDrive/soil_microCT_images/nnUNet_resources/bnei_reem'

nnUNet_raw = os.path.join(DRIVE_BASE, 'nnUNet_raw')
nnUNet_preprocessed = os.path.join(DRIVE_BASE, 'nnUNet_preprocessed')
nnUNet_results = os.path.join(DRIVE_BASE, 'nnUNet_results')

os.environ['nnUNet_raw'] = nnUNet_raw
os.environ['nnUNet_preprocessed'] = nnUNet_preprocessed
os.environ['nnUNet_results'] = nnUNet_results

for d in [nnUNet_raw, nnUNet_preprocessed, nnUNet_results]:
    os.makedirs(d, exist_ok=True)

print('nnUNet_raw:', nnUNet_raw)
print('nnUNet_preprocessed:', nnUNet_preprocessed)
print('nnUNet_results:', nnUNet_results)

## 4) Upload Training Data

The local step `preprocessing_nnUNet_train.py` must have produced:
```
Dataset777_GCEF/
  imagesTr/*_0000.nii.gz
  labelsTr/*.nii.gz
  dataset.json
```
This folder should already exist on Drive (created locally via junction).

In [ ]:
# Verify uploaded dataset
dataset_dir = os.path.join(nnUNet_raw, 'Dataset777_GCEF')
assert os.path.isdir(dataset_dir), f'Dataset folder not found: {dataset_dir}'

import glob
images = glob.glob(os.path.join(dataset_dir, 'imagesTr', '*_0000.nii.gz'))
labels = glob.glob(os.path.join(dataset_dir, 'labelsTr', '*.nii.gz'))
dataset_json = os.path.join(dataset_dir, 'dataset.json')

print(f'imagesTr: {len(images)} files')
print(f'labelsTr: {len(labels)} files')
print(f'dataset.json exists: {os.path.isfile(dataset_json)}')
assert len(images) > 0, 'No training images found'
assert len(labels) > 0, 'No training labels found'
assert os.path.isfile(dataset_json), 'dataset.json missing'

## 5) nnUNet Planning & Preprocessing

In [ ]:
!nnUNetv2_plan_and_preprocess -d 777 --verify_dataset_integrity

## 6) Training

Run one fold at a time. Change `FOLD` to train multiple folds sequentially (0–4).

**Warning**: each fold may take many hours on a single Colab GPU.

In [ ]:
# Fix PolyLRScheduler for PyTorch 2.6+ (verbose param removed from LRScheduler)
from torch.optim.lr_scheduler import LRScheduler
import nnunetv2.training.lr_scheduler.polylr as polylr_module

class PolyLRScheduler(LRScheduler):
    def __init__(self, optimizer, initial_lr, max_steps, exponent=0.9, current_step=None):
        self.optimizer = optimizer
        self.initial_lr = initial_lr
        self.max_steps = max_steps
        self.exponent = exponent
        self.ctr = 0
        super().__init__(optimizer, current_step if current_step is not None else -1)

    def step(self, current_step=None):
        if current_step is not None:
            self.ctr = current_step
        for param_group, lr in zip(self.optimizer.param_groups, self.get_lr()):
            param_group['lr'] = lr
        self.ctr += 1

    def get_lr(self):
        return [self.initial_lr * (1 - self.ctr / self.max_steps) ** self.exponent
                for _ in self.optimizer.param_groups]

polylr_module.PolyLRScheduler = PolyLRScheduler
print("PolyLRScheduler patched for PyTorch 2.6+ compatibility")

In [ ]:
# Create custom splits_final.json (single sample used for both train and val)
import json, os

preprocessed_dir = os.path.join(os.environ['nnUNet_preprocessed'], 'Dataset777_GCEF')
splits = [{"train": ["nlm_volume"], "val": ["nlm_volume"]}]

splits_path = os.path.join(preprocessed_dir, 'splits_final.json')
with open(splits_path, 'w') as f:
    json.dump(splits, f, indent=2)

print(f"Wrote {splits_path}")
print(json.dumps(splits, indent=2))

In [ ]:
# Only fold 0 — single split entry in splits_final.json
!nnUNetv2_train 777 3d_fullres 0 -tr nnUNetTrainer_betterIgnoreSampling

In [ ]:
# List training results
results_dir = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    'nnUNetTrainer_betterIgnoreSampling__nnUNetPlans__3d_fullres'
)
if os.path.isdir(results_dir):
    for item in sorted(os.listdir(results_dir)):
        print(item)
else:
    print(f'Results dir not found yet: {results_dir}')

## 7) Download Training Outputs

Training logs and checkpoints are saved in:
```
{nnUNet_results}/Dataset777_GCEF/nnUNetTrainer_betterIgnoreSampling__nnUNetPlans__3d_fullres/
```
Download `training_log*`, `checkpoint_best.pth`, `checkpoint_final.pth`, and `plans.json` for local analysis.

In [ ]:
# Optional: zip results for download
import shutil
results_dir = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    'nnUNetTrainer_betterIgnoreSampling__nnUNetPlans__3d_fullres'
)
if os.path.isdir(results_dir):
    archive_path = '/content/training_results_777'
    shutil.make_archive(archive_path, 'zip', results_dir)
    print(f'Archive: {archive_path}.zip')
else:
    print('No results to archive yet.')

## 8) Upload Inference Data

Run locally first:
1. `preprocessing_nnUNet_predict_tif.py` → `*_0000.nii.gz`
2. `preprocessing_nnUNet_predict_split.py` → split chunks `sample__axis__min__max__0000.nii.gz`

Upload split chunk folders to Drive.

In [ ]:
# === TODO: edit these paths ===
INFERENCE_INPUT = os.path.join(DRIVE_BASE, 'inference_input')   # folder with split *_0000.nii.gz chunks
INFERENCE_OUTPUT = os.path.join(DRIVE_BASE, 'inference_output')  # output predictions
os.makedirs(INFERENCE_OUTPUT, exist_ok=True)

assert os.path.isdir(INFERENCE_INPUT), f'Inference input not found: {INFERENCE_INPUT}'
input_files = [f for f in os.listdir(INFERENCE_INPUT) if f.endswith('_0000.nii.gz')]
print(f'Inference chunks found: {len(input_files)}')

## 9) Inference

In [ ]:
!nnUNetv2_predict \
    -i {INFERENCE_INPUT} \
    -o {INFERENCE_OUTPUT} \
    -d 777 \
    -tr nnUNetTrainer_betterIgnoreSampling \
    -c 3d_fullres

## 10) Validate & Download Predictions

In [ ]:
# List prediction outputs
pred_files = [f for f in os.listdir(INFERENCE_OUTPUT) if f.endswith('.nii.gz')]
print(f'Predictions: {len(pred_files)} files')
for f in sorted(pred_files):
    print(f'  {f}')

In [ ]:
# Optional: zip predictions for download
import shutil
if pred_files:
    archive_path = '/content/predictions_777'
    shutil.make_archive(archive_path, 'zip', INFERENCE_OUTPUT)
    print(f'Archive: {archive_path}.zip')
else:
    print('No predictions to archive.')